## Functional QUBO Solver Tutorial (step-by-step pipeline)

This notebook is a *code-first* walkthrough of the **functional** composition style in `qubo-solver`: you build an end-to-end solver by explicitly chaining small, well-named “LEGO brick” functions (transforms → embedding → drive shaping → solvers), rather than relying on a single opaque `solve()` entry point.

The goal is twofold:

- **Transparency:** every line corresponds to a concrete algorithmic step, so it’s always clear *what ran*, *in what order*, and *which artifact* each step produced.
- **Composability:** once you understand the contracts (inputs/outputs) of each step, you can swap components (different embeddings, different drive shaping strategies, different solvers) without rewriting everything.

## Setup and Imports

In [ ]:
from __future__ import annotations

import numpy as np
import random
import torch
from typing import Literal
import warnings

import qoolqit

from qubosolver import (
    Instance,
    Solution,
    SingleSolution,
    analysis,
    solvers,
    transforms,
    embedding,
    drive_shaping,
    torch_rng,
    linalg,
    tensor,
)

## Utility Functions

In [2]:
def gather_optimal_solutions(solutions: Solution) -> list[SingleSolution]:
    """Find all solutions with minimum cost."""
    min_cost = solutions[0].cost
    return [d for d in solutions if np.allclose(d.cost, min_cost)]

def manual_seed(seed: int) -> torch.Generator:
    """Set random seeds for reproducibility."""
    np.random.seed(seed)
    torch.manual_seed(seed)
    random.seed(seed)
    return torch_rng(seed)

def interaction_matrix_from_vertices(vertices: torch.Tensor) -> torch.Tensor:
    """Create interaction matrix based on vertex distances."""
    U = 1.0 / torch.cdist(vertices, vertices) ** 6
    U.fill_diagonal_(0.0)
    return U

## Creating a Simple QUBO Problem

We'll create a simple QUBO instance based on geometric vertices with interaction strengths.

In [3]:
def create_simple_qubo():
    """Create a simple QUBO problem and find its optimal solutions."""
    
    # Define vertices in 2D space
    sqrt3 = np.sqrt(3.0)
    vertices = tensor.tensor([
        [0.0, 0.0],
        [-1.0, 0.0],
        [-1.5, -0.5 * sqrt3],
        [-0.5, -0.5 * sqrt3],
        [4.0, 0.0],
    ])
    
    n_qubits = vertices.shape[0]
    
    # Create QUBO matrix from vertex interactions
    diagonal_scale = -2.0
    diagonal = torch.ones(n_qubits, dtype=linalg.dtype())
    Q = interaction_matrix_from_vertices(vertices) + diagonal_scale * torch.diag(diagonal)
    Q /= Q.max()  # Normalize
    
    # Find optimal solutions by exhaustive search
    solutions = solvers.brute_force(Instance(Q))
    
    expected_optimal_solutions = gather_optimal_solutions(solutions)
    
    print(f"QUBO Matrix Shape: {Q.shape}")
    print(f"Expected Minimum cost: {expected_optimal_solutions[0].cost:.6f}")
    print(f"Number of optimal solutions: {len(expected_optimal_solutions)}")
    print(f"Optimal bitstrings: {[s.string for s in expected_optimal_solutions]}")
    
    return Instance(Q), expected_optimal_solutions

# Create the QUBO problem
seed = 16844214
manual_seed(seed)
qubo, expected_optimal_solutions = create_simple_qubo()

QUBO Matrix Shape: torch.Size([5, 5])
Expected Minimum cost: -5.925371
Number of optimal solutions: 1
Optimal bitstrings: ['10101']


## Solution Analysis Function

In [4]:
def analyze_solution(solutions: Solution, expected_optimal_solutions: list[SingleSolution], 
                    expect_optimality: bool = True):
    """Analyze and validate QUBO solutions."""
    
    # Check for duplicate solutions
    unique_count = solutions.bitstrings.unique(dim=0).shape[0]
    total_count = len(solutions)
    print(f"Solutions are unique: {unique_count == total_count}")
    
    # Create analyzer for detailed statistics
    df = analysis.to_dataframe([solutions])
    print(f"\nSolution Statistics:\n{df}")
    
    # Find optimal solutions
    optimal_solutions = gather_optimal_solutions(solutions)
    min_cost = optimal_solutions[0].cost
    
    print(f"\nFound minimum cost: {min_cost:.6f}")
    print(f"Found optimal bitstrings: {[s.string for s in optimal_solutions]}")
    print(f"Number of found optimal solutions: {len(optimal_solutions)}")
    
    if expect_optimality:
        # Check if we found the true optimum
        expected_cost = expected_optimal_solutions[0].cost
        cost_match = np.allclose(min_cost, expected_cost)
        print("\nOptimality check:")
        print(f"Expected cost: {expected_cost:.6f}")
        print(f"Cost matches expected: {cost_match}")
        
        # Check solution quality
        cumulated_probability = sum(s.probability for s in optimal_solutions)
        print(f"Total probability of optimal solutions: {cumulated_probability:.4f}")
        print(f"Good solution quality (>75%): {cumulated_probability > 0.75}")
    
    return optimal_solutions

## Quantum Solving Workflow

Now let's solve the QUBO problem using quantum methods with different configurations.

In [5]:
"""Solve QUBO using quantum methods."""

embedding_method: Literal["blade", "greedy"] = "blade"
drive_shaping_method: Literal["proportional_diagonal", "bayesian_search"] = "proportional_diagonal"
preprocessing: bool = True
postprocessing: bool = True

seed = 16844214
manual_seed(seed)

warnings.filterwarnings("ignore")

print("\n=== Quantum Solving ===")
print(f"Embedding: {embedding_method}, Drive shaping: {drive_shaping_method}")
print(f"Preprocessing: {preprocessing}, Postprocessing: {postprocessing}")

# Check for trivial solutions
trivial_solution = solvers.trivial_solution_search(qubo)
print(f"Trivial solution found: {bool(trivial_solution)}")

effective_qubo = qubo
if preprocessing:
    print("Applying variable fixing preprocessing...")
    effective_qubo = transforms.variable_fixing.apply_recursively(qubo)

# Set up quantum device
device = qoolqit.AnalogDevice()

# Embedding
print(f"Embedding with {embedding_method} method...")
if embedding_method == "blade":
    blade_config = embedding.blade.Config(device=device)
    register = embedding.blade.embed(effective_qubo, config=blade_config)
elif embedding_method == "greedy":
    greedy_config = embedding.greedy.Config(traps=100)
    register = embedding.greedy.embed(effective_qubo, device, config=greedy_config)
else:
    raise ValueError(f"Invalid embedding method: {embedding_method}")

print(f"Register qubits: {register.qubits}")
print(f"Register distances: {register.distances()}")

# Set up emulator
emulator = qoolqit.execution.LocalEmulator()

# Drive shaping
print(f"Generating drive with {drive_shaping_method} method...")
if drive_shaping_method == "proportional_diagonal":
    drive = drive_shaping.proportional_diagonal.build_drive(
        effective_qubo, register, device=device, dmm=False, kappa=0.25
    )
elif drive_shaping_method == "bayesian_search":
    drive, _ = drive_shaping.bayesian_search.build_drive(
        effective_qubo, register, emulator, device, dmm=False,
        config=drive_shaping.bayesian_search.Config(n_calls=11, seed=seed)
    )
else:
    raise ValueError(f"Invalid drive shaping method: {drive_shaping_method}")

# Execute quantum sampling
print("Running quantum sampling...")
job = solvers.analog_quantum_sampling(register, drive, emulator, device)
solution = Solution.from_results(job.results())

# Post-process fixations and restore original QUBO
if preprocessing:
    print("Unapplying preprocessing transformations...")
    assert isinstance(effective_qubo, transforms.variable_fixing.Instance)
    solution = transforms.variable_fixing.lift(solution, effective_qubo)

if postprocessing:
    print("Applying local search postprocessing...")
    solution = solvers.iterative_bitflip_local_search(qubo, solution)
    
analyze_solution(solution, expected_optimal_solutions)



=== Quantum Solving ===
Embedding: blade, Drive shaping: proportional_diagonal
Preprocessing: True, Postprocessing: True
Trivial solution found: False
Applying variable fixing preprocessing...
Embedding with blade method...
Register qubits: {'0': array([ 8.66025404e-01, -1.66925145e-16]), '1': array([-2.79487192e-16, -5.00000000e-01]), '2': array([-8.66025404e-01,  1.66925145e-16]), '3': array([2.79487192e-16, 5.00000000e-01])}
Register distances: {('0', '3'): 0.9999999999999998, ('0', '2'): 1.7320508075688772, ('2', '3'): 1.0000000000000002, ('1', '3'): 1.0, ('1', '2'): 0.9999999999999998, ('0', '1'): 1.0000000000000002}
Generating drive with proportional_diagonal method...
Running quantum sampling...
Unapplying preprocessing transformations...
Applying local search postprocessing...
Solutions are unique: True

Solution Statistics:
   labels bitstrings     costs  counts  probs
0       0      10101 -5.925371     798  0.798
1       0      00101 -3.999933      16  0.016
2       0      0

[SingleSolution(bitstring=tensor([1, 0, 1, 0, 1], dtype=torch.int8), cost=-5.9253705103064815, count=798, probability=0.7979999780654907)]

## Classical Solving Methods

Let's also solve the same problem using classical optimization methods.

In [6]:
"""Solve QUBO using classical methods."""

solving_method: Literal["cplex", "tabu", "sa", "sa+tabu", "random"] = "cplex"
preprocessing: bool = True
postprocessing: bool = True

seed = 16844214
rng = manual_seed(seed)
    
print("\n=== Classical Solving ===")
print(f"Method: {solving_method}")
print(f"Preprocessing: {preprocessing}, Postprocessing: {postprocessing}")

rng = torch_rng(seed)

# Check for trivial solutions
trivial_solution = solvers.trivial_solution_search(qubo)
print(f"Trivial solution found: {bool(trivial_solution)}")

effective_qubo = qubo
if preprocessing:
    print("Applying variable fixing preprocessing...")
    effective_qubo = transforms.variable_fixing.apply_recursively(qubo)

# Solve using the specified method
print(f"Solving with {solving_method}...")
if solving_method == "cplex":
    solution = solvers.cplex(effective_qubo)
elif solving_method == "tabu":
    initial_solution = solvers.random_sampling(effective_qubo, rng=rng, max_bitstrings=1)
    solution = solvers.tabu_search(effective_qubo, initial_solution.bitstrings)
elif solving_method == "sa":
    solution = solvers.simulated_annealing(effective_qubo, rng=rng, top_k=1)
elif solving_method == "sa+tabu":
    sa_solution = solvers.simulated_annealing(effective_qubo, rng=rng, top_k=1)
    solution = solvers.tabu_search(effective_qubo, sa_solution.bitstrings)
elif solving_method == "random":
    solution = solvers.random_sampling(effective_qubo, rng=rng)
else:
    raise ValueError(f"Invalid solving method: {solving_method}")

# Post-process
if preprocessing:
    print("Unapplying preprocessing transformations...")
    assert isinstance(effective_qubo, transforms.variable_fixing.Instance)
    solution = transforms.variable_fixing.lift(solution, effective_qubo)

if postprocessing:
    print("Applying local search postprocessing...")
    solution = solvers.iterative_bitflip_local_search(qubo, solution)

expect_optimality = solving_method != "random"
analyze_solution(solution, expected_optimal_solutions, expect_optimality)
    


=== Classical Solving ===
Method: cplex
Preprocessing: True, Postprocessing: True
Trivial solution found: False
Applying variable fixing preprocessing...
Solving with cplex...
Unapplying preprocessing transformations...
Applying local search postprocessing...
Solutions are unique: True

Solution Statistics:
  labels bitstrings     costs  counts  probs
0      0      10101 -5.925371       1    1.0

Found minimum cost: -5.925371
Found optimal bitstrings: ['10101']
Number of found optimal solutions: 1

Optimality check:
Expected cost: -5.925371
Cost matches expected: True
Total probability of optimal solutions: 1.0000
Good solution quality (>75%): True


[SingleSolution(bitstring=tensor([1, 0, 1, 0, 1], dtype=torch.int8), cost=-5.9253705103064815, count=1, probability=1.0)]